# Run a Reusable Scientific Composition

A composite is one outer pipeline step. It calls creators or other composites locally, and exports only its declared final outputs plus required semantic support. Curators such as Filter remain outer pipeline steps.

These examples are installed with Artisan, so a fresh worker can import and verify their definitions. Inline authoring is covered in [Writing a Composite](../09-writing-operations/02-writing-a-composite.ipynb).

In [ ]:
from __future__ import annotations

from artisan.composites.examples import GenerateAndScore, TransformAndScore
from artisan.inspection import inspect_evidence
from artisan.operations.examples import DataGenerator, DataTransformer, MetricCalculator
from artisan.orchestration import PipelineManager
from artisan.storage import ArtifactStore
from artisan.utils import tutorial_setup
from artisan.visualization import (
    build_macro_graph,
    build_micro_graph,
    inspect_pipeline,
    inspect_step,
)


def contents(env, run, step, artifact_type):
    refs = inspect_evidence(
        "query_artifacts",
        {"pipeline_run_id": run, "artifact_type": artifact_type, "limit": 100},
        delta_root=env.delta_root,
    )
    ids = [item.artifact_id for item in refs.items if item.current_step_number == step]
    return {
        a.content
        for a in ArtifactStore(env.delta_root)
        .get_artifacts_by_type(ids, artifact_type)
        .values()
    }

In [ ]:
from functools import partial
from tempfile import TemporaryDirectory

# Keep this tutorial's stores and source files in one owned temporary workspace.
workspace = TemporaryDirectory(prefix="artisan-tutorial-")
tutorial_setup = partial(tutorial_setup, base_dir=workspace.name)

## Separate pipeline steps

First run generation, transformation and scoring as three ordinary steps.

In [ ]:
baseline_env = tutorial_setup("composite_baseline")
with PipelineManager.create(
    name="baseline",
    delta_root=baseline_env.delta_root,
    staging_root=baseline_env.staging_root,
    working_root=baseline_env.working_root,
) as baseline:
    generated = baseline.run(DataGenerator, params={"count": 3, "seed": 42})
    transformed = baseline.run(
        DataTransformer,
        name="transform",
        inputs={"dataset": generated.output("datasets")},
        params={"scale_factor": 2.0, "variants": 1, "seed": 100},
    )
    baseline.run(
        MetricCalculator,
        name="score",
        inputs={"dataset": transformed.output("dataset")},
    )
    baseline_summary = baseline.finalize()
assert baseline_summary["overall_success"]
assert baseline_summary["total_steps"] == 3
inspect_pipeline(baseline_env.delta_root)

## One composite boundary

`TransformAndScore` calls DataTransformer then MetricCalculator. Both final datasets and their directly associated metrics are exported. Child invocations appear in execution diagnostics; they are not extra pipeline steps.

In [ ]:
env = tutorial_setup("composite_boundary")
with PipelineManager.create(
    name="composition",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
) as pipeline:
    generated = pipeline.run(DataGenerator, params={"count": 3, "seed": 42})
    composed = pipeline.run(
        TransformAndScore,
        name="transform_score",
        inputs={"dataset": generated.output("datasets")},
        params={"scale_factor": 2.0},
        batch_strategy={"artifacts_per_unit": 2},
    )
    summary = pipeline.finalize()
assert summary["overall_success"]
assert summary["total_steps"] == 2
assert contents(
    env, pipeline.config.pipeline_run_id, composed.step_number, "csv"
) == contents(baseline_env, baseline.config.pipeline_run_id, 1, "csv")
assert inspect_step(env.delta_root, composed.step_number).height == 6
inspect_pipeline(env.delta_root)

The root `artifacts_per_unit` selects outer scientific partitions. Operations and composites can declare defaults for these units, worker packing, and resources; supplied submission fields override them. Creator children keep their own invocation policy. Nested composites expand within the same boundary without reapplying their outer defaults.


In [ ]:
build_macro_graph(env.delta_root)

In [ ]:
build_micro_graph(env.delta_root)

## Nest a composition

`GenerateAndScore` generates data inside the outer boundary, then calls TransformAndScore. The generated intermediate datasets are not exported. Final data and its metrics remain available together. Nesting creates no additional worker allocation, persistence boundary, or cache entry.


In [ ]:
nested_env = tutorial_setup("composite_nested")
with PipelineManager.create(
    name="nested",
    delta_root=nested_env.delta_root,
    staging_root=nested_env.staging_root,
    working_root=nested_env.working_root,
) as nested:
    result = nested.run(
        GenerateAndScore, params={"count": 2, "scale_factor": 1.5, "seed": 42}
    )
    summary = nested.finalize()
assert summary["overall_success"]
assert summary["total_steps"] == 1
assert inspect_step(nested_env.delta_root, result.step_number).height == 4
inspect_pipeline(nested_env.delta_root)

In [ ]:
build_micro_graph(nested_env.delta_root)